# Visualize the agent's memory graph

An interactive view of everything the agent remembers, built with
[`pyvis`](https://pyvis.readthedocs.io/) and rendered as a self-contained
`<iframe srcdoc>` with vis.js **inlined**. That means it needs *no* Jupyter
widget frontend extension (no `GraphModel` / ipywidgets manager), so it
renders reliably in JupyterLab and Notebook 7, online or offline. Pan, zoom,
drag nodes, and hover to see each node's full property set.

Node labels (see the schema sketch in the README): `Message`, `GameSnapshot`,
reasoning `Trace` / `Step` / `ToolCall`, and the long-term `Entity` /
`Preference` semantic model.

Prereqs:
- Neo4j is up (`bash scripts/vast_neo4j_launch.sh`, or `docker compose up -d neo4j`).
- `pip install -r requirements.txt` (installs `pyvis`).

In [1]:
import os
import sys

# Run from the repo root so `agent` imports and `.env` resolve.
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())

from IPython.display import HTML, display
from neo4j import GraphDatabase
from pyvis.network import Network

from agent.config import CONFIG

driver = GraphDatabase.driver(
    CONFIG.neo4j_uri,
    auth=(CONFIG.neo4j_username, CONFIG.neo4j_password),
    database=CONFIG.neo4j_database,
)

# Color + caption per node label; the caption fn receives the property dict.
LABEL_STYLE = {
    "Message": ("#4C78A8", lambda p: f"{p.get('role', '?')}: {str(p.get('content', ''))[:40]}"),
    "GameSnapshot": ("#F58518", lambda p: f"snapshot [{p.get('label', 'step')}]"),
    "Trace": ("#54A24B", lambda p: f"Trace: {str(p.get('task', ''))[:40]}"),
    "Step": ("#88D27A", lambda p: f"Step: {str(p.get('thought', ''))[:40]}"),
    "ToolCall": ("#B79A20", lambda p: f"tool: {p.get('tool', '')}"),
    "Entity": ("#E45756", lambda p: f"{p.get('name', '')} ({p.get('type', '')})"),
    "Preference": ("#72B7B2", lambda p: f"pref: {p.get('category', '')}"),
}


def _style(labels, props):
    for lbl in labels:
        if lbl in LABEL_STYLE:
            color, textfn = LABEL_STYLE[lbl]
            return color, textfn(props), lbl
    first = labels[0] if labels else "node"
    return "#BAB0AC", first, first


def _add_node(net, node, seen):
    nid = node.element_id
    if nid in seen:
        return
    seen.add(nid)
    props = dict(node)
    labels = list(node.labels)
    color, text, lbl = _style(labels, props)
    # Hover tooltip = every property. Skip the huge base64 thumbnail blob, and
    # truncate generously (not at 80 chars -- that was cutting long values like
    # the `controls` tip off mid-sentence, e.g. hiding FORWARD).
    _skip = {"thumbnail_b64"}
    tip = lbl + "\n" + "\n".join(
        f"{k}: {str(v)[:1000]}" for k, v in props.items() if k not in _skip
    )
    net.add_node(nid, label=text, color=color, title=tip)


def render_cypher(query, params=None, height="640px"):
    """Run a Cypher query and return an interactive, self-contained graph.

    Rendered as an <iframe srcdoc=...> with vis.js inlined -- no Jupyter
    widget frontend extension required.
    """
    net = Network(
        height=height, width="100%", directed=True, notebook=False,
        cdn_resources="in_line", bgcolor="#ffffff", font_color="#222222",
    )
    net.barnes_hut()
    seen = set()
    with driver.session(database=CONFIG.neo4j_database) as s:
        for rec in s.run(query, **(params or {})):
            rel = rec.get("r")
            if rel is not None:
                _add_node(net, rel.start_node, seen)
                _add_node(net, rel.end_node, seen)
                net.add_edge(
                    rel.start_node.element_id, rel.end_node.element_id,
                    label=rel.type, title=rel.type,
                )
            for key in ("n", "m"):
                node = rec.get(key)
                if node is not None:
                    _add_node(net, node, seen)
    doc = net.generate_html(notebook=False)
    # Escape only what an srcdoc attribute needs (& and "), so the inner
    # HTML still parses and its scripts run in the nested document.
    srcdoc = doc.replace("&", "&amp;").replace('"', "&quot;")
    return HTML(
        f'<iframe srcdoc="{srcdoc}" width="100%" height="680" '
        f'style="border:1px solid #ddd; border-radius:6px;"></iframe>'
    )

## The whole memory graph

Everything currently stored (isolated nodes included). If the graph is large,
raise/lower the `LIMIT`.

In [3]:
render_cypher("MATCH (n) OPTIONAL MATCH (n)-[r]->(m) RETURN n, r, m LIMIT 500")

## One conversation / session

Scroll through a single session's experiences: its messages, the game
snapshots they captured, and the reasoning traces behind each move. Set
`SESSION_ID` to the `session_id` printed by the play notebook or `agent.runner`.

In [ ]:
SESSION_ID = ""  # e.g. the session_id printed by notebooks/play.ipynb

if SESSION_ID:
    display(render_cypher(
        "MATCH (n {session_id: $sid}) "
        "OPTIONAL MATCH (n)-[r]->(m) "
        "RETURN n, r, m",
        {"sid": SESSION_ID},
    ))
else:
    print("Set SESSION_ID above to scope the view to one conversation.")

When you're done:

In [ ]:
driver.close()